# QTrend_v2 v1 walkthrough

End-to-end demo: load HC data, run one bias window, render report.


In [ ]:
import pandas as pd
from qtrend_v2 import Strategy
from qtrend_v2.bias import BiasWindow, BiasWindowLoader
from qtrend_v2.data import load_hc_daily, load_hc_1h
from qtrend_v2.report import render_window_report, render_aggregate_report
from pathlib import Path


## 1. Load HC data and bias windows

In [ ]:
daily = load_hc_daily()
h1    = load_hc_1h()
print(f'daily: {len(daily)} bars, {daily.index.min().date()} -> {daily.index.max().date()}')
print(f'1H   : {len(h1)} bars')


In [ ]:
loader = BiasWindowLoader(Path('..') / 'data' / 'bias_windows.csv')
for w in loader.windows():
    print(w.start.date(), '->', w.end.date(), '|', w.note)


## 2. Run strategy on one window

In [ ]:
strat = Strategy()
windows = [w for w in loader.windows() if w.start >= daily.index.min()]
w = windows[0]
result = strat.run_window(window=w, daily=daily, h1=h1)
result.actions_log.head(20)


## 3. Inspect equity, lot history, forecast

In [ ]:
result.equity.plot(title='Equity', figsize=(10, 3))


In [ ]:
result.lot_history.plot(title='Lot history', figsize=(10, 2), drawstyle='steps-post')


In [ ]:
result.forecast_history.plot(title='Daily forecast', figsize=(10, 3), color='purple')


## 4. Render reports

In [ ]:
reports_dir = Path('..') / 'reports'
reports_dir.mkdir(exist_ok=True)
render_window_report(result=result, daily=daily, output_path=reports_dir / f'window_{w.start.date()}.html')


In [ ]:
results = [strat.run_window(window=w_, daily=daily, h1=h1) for w_ in windows]
render_aggregate_report(results=results, output_path=reports_dir / 'aggregate.html')
print('Reports:', list(reports_dir.glob('*.html')))


## Next steps

- Replace `bias_windows.csv` placeholders with real annotations.
- Swap `EWMAC()` for `TSMOM` or `DonchianFraction` and re-run.
- Compare aggregate hit rate / PnL distribution across forecasts.
